In [1]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.eda_utils import parse_money, binarize_target

FIG = ROOT / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
CSV = ROOT / "SBAnational.csv"
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")
print("csv exists:", CSV.exists())

csv exists: True


In [2]:
MONEY_COLS = ["DisbursementGross", "BalanceGross", "ChgOffPrinGr", "GrAppv", "SBA_Appv"]
df = pd.read_csv(CSV, dtype=str, low_memory=False)
for c in MONEY_COLS:
    df[c] = parse_money(df[c])
for c in ["Term", "NoEmp", "CreateJob", "RetainedJob", "ApprovalFY", "FranchiseCode", "UrbanRural"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["ApprovalDate"] = pd.to_datetime(df["ApprovalDate"], errors="coerce", format="%d-%b-%y")
df["target"] = binarize_target(df["MIS_Status"])
print("shape:", df.shape)
df.head(3)

shape: (899164, 28)


,LoanNr_ChkDgt,Name,City,State,Zip,Bank,BankState,NAICS,ApprovalDate,ApprovalFY,Term,NoEmp,NewExist,CreateJob,RetainedJob,FranchiseCode,UrbanRural,RevLineCr,LowDoc,ChgOffDate,DisbursementDate,DisbursementGross,BalanceGross,MIS_Status,ChgOffPrinGr,GrAppv,SBA_Appv,target
0,1000014003,ABC HOBBYCRAFT,EVANSVILLE,IN,47711,FIFTH THIRD BANK,OH,451120,1997-02-28,1997.0,84,4,2,0,0,1,0,N,Y,NaN,28-Feb-99,60000.0,0.0,P I F,0.0,60000.0,48000.0,0.0
1,1000024006,LANDMARK BAR & GRILLE (THE),NEW PARIS,IN,46526,1ST SOURCE BANK,IN,722410,1997-02-28,1997.0,60,2,2,0,0,1,0,N,Y,NaN,31-May-97,40000.0,0.0,P I F,0.0,40000.0,32000.0,0.0
2,1000034009,"WHITLOCK DDS, TODD M.",BLOOMINGTON,IN,47401,GRANT COUNTY STATE BANK,IN,621210,1997-02-28,1997.0,180,7,1,0,0,1,0,N,N,NaN,31-Dec-97,287000.0,0.0,P I F,0.0,287000.0,215250.0,0.0


In [3]:
# NewExist 공식 정의: 1 = Existing business, 2 = New business
print(df["NewExist"].value_counts(dropna=False))
# 이상치(0)와 결측은 분석에서 제외 표시
df["is_new"] = df["NewExist"].map({"1": 0, "2": 1})  # 1=신규(New)
print("\nis_new(1=신규) 분포:")
print(df["is_new"].value_counts(dropna=False))

NewExist
1      644869
2      253125
0        1034
NaN       136
Name: count, dtype: int64

is_new(1=신규) 분포:
is_new
0.0    644869
1.0    253125
NaN      1170
Name: count, dtype: int64


In [4]:
chk = df.dropna(subset=["target"]).groupby("NewExist")["target"].agg(["mean", "size"])
print("NewExist별 부실률(target mean) / 건수:")
print(chk)

NewExist별 부실률(target mean) / 건수:
              mean    size
NewExist                  
0         0.061284    1028
1         0.171132  643446
2         0.187548  252559


In [5]:
miss = df.isna().mean().sort_values(ascending=False)
print("결측률 상위:")
print((miss[miss > 0] * 100).round(2).astype(str) + "%")


결측률 상위:
ChgOffDate          81.91%
RevLineCr             0.5%
LowDoc               0.29%
DisbursementDate     0.26%
target               0.22%
MIS_Status           0.22%
BankState            0.17%
Bank                 0.17%
is_new               0.13%
NewExist             0.02%
City                  0.0%
ApprovalFY            0.0%
Name                  0.0%
State                 0.0%
dtype: object


In [6]:
num_cols = ["GrAppv", "SBA_Appv", "Term", "NoEmp", "CreateJob", "RetainedJob"]
display(df[num_cols].describe(percentiles=[.01, .5, .99]).T)
print("Term==0:", (df["Term"] == 0).sum())
print("GrAppv<=0:", (df["GrAppv"] <= 0).sum())
print("NoEmp==0:", (df["NoEmp"] == 0).sum())


,count,mean,std,min,1%,50%,99%,max
GrAppv,899164.0,192686.976384,283263.391297,200.0,5000.0,90000.0,1350000.0,5472000.0
SBA_Appv,899164.0,149488.788175,228414.561519,100.0,2500.0,61250.0,1000000.0,5472000.0
Term,899164.0,110.773078,78.857305,0.0,5.0,84.0,300.0,569.0
NoEmp,899164.0,11.411353,74.108196,0.0,1.0,4.0,95.0,9999.0
CreateJob,899164.0,8.430376,236.688165,0.0,0.0,0.0,30.0,8800.0
RetainedJob,899164.0,10.797257,237.120600,0.0,0.0,1.0,55.0,9500.0


Term==0: 810
GrAppv<=0: 0
NoEmp==0: 6631
